In [ ]:
!pip install datasets

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from datasets import load_dataset

In [3]:
dataset = load_dataset("amazon_polarity")

train_data = dataset["train"].select(range(20000))
test_data = dataset["test"].select(range(5000))

In [4]:
train_data[0]

{'label': 1,
 'title': 'Stuning even for the non-gamer',
 'content': 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^'}

In [5]:
def tokenize(text):
    return text.lower().split()

In [6]:
from collections import Counter
def build_vocab(texts, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab([sample["content"] for sample in train_data], min_freq=2)

In [7]:
from collections import Counter

def build_vocab(texts, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<pad>": 0, "<unk>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

In [8]:
vocab = build_vocab([sample["content"] for sample in train_data], min_freq=2)
vocab_size = len(vocab)
vocab_size

35366

In [9]:
vocab_size

35366

In [10]:
max_len = 100

def encode(text):
    tokens = tokenize(text)
    seq = [vocab.get(word, 1) for word in tokens]
    return seq[:max_len]

def pad(seq):
    return seq + [0]*(max_len - len(seq))

In [11]:
def encode(text, vocab, max_len = 100):
    return [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)][:max_len]

def pad(sequences, max_len):
    padded = []

    for seq in sequences:
        seq = seq[:max_len]
        seq = seq + [0] * (max_len - len(seq))  # pad=0
        padded.append(seq)

    return padded

In [12]:
class AmazonDataset(torch.utils.data.Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        text = self.data[idx]["content"]
        label = self.data[idx]["label"]

        seq = pad([encode(text, vocab)], max_len)[0]

        return torch.tensor(seq), torch.tensor(label)

In [13]:
train_loader = DataLoader(
    AmazonDataset(train_data),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    AmazonDataset(test_data),
    batch_size=64
)

In [14]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        # x: (batch, seq_len)

        x = self.embedding(x)
        # (batch, seq_len, embed_dim)

        output, hidden = self.rnn(x)
        # hidden: (1, batch, hidden_dim)

        return self.fc(hidden.squeeze(0))

In [15]:
class BetterRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.rnn = nn.RNN(
            embed_dim,
            hidden_dim,
            num_layers=2,
            nonlinearity='relu',
            batch_first=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        out = torch.mean(output, dim=1)

        out = self.dropout(out)

        return self.fc(out)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
simple_model = SimpleRNN(
    vocab_size= vocab_size,
    embed_dim=100,
    hidden_dim=128
).to(device)
model = BetterRNN(
    vocab_size= vocab_size,
    embed_dim=100,
    hidden_dim=128
).to(device)

Using device: cuda


In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [19]:
epochs = 10

for epoch in range(epochs):

    for text, label in train_loader:

        text = text.to(device)
        label = label.to(device)

        outputs = model(text)

        loss = criterion(outputs, label)

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.3724
Epoch 2, Loss: 0.3045
Epoch 3, Loss: 0.3176
Epoch 4, Loss: 0.2593
Epoch 5, Loss: 0.1366
Epoch 6, Loss: 0.1072
Epoch 7, Loss: 0.1035
Epoch 8, Loss: 0.0960
Epoch 9, Loss: 0.0067
Epoch 10, Loss: 0.0289


In [59]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = simple_model(texts)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 48.46


In [20]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts) #better rnn model

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 79.62


In [23]:
def predict(text):

    model.eval()

    with torch.no_grad():

        seq = pad([encode(text, vocab)], max_len)[0]
        seq = torch.tensor(seq).unsqueeze(0).to(device)

        output = model(seq)

        _, pred = torch.max(output,1)

        return "Positive" if pred.item() == 1 else "Negative"

In [24]:
print(predict("this product is amazing"))
print(predict("don't waste your money on this"))

Positive
Negative
